# 1. Imports

In [7]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# 2. Connection

In [10]:
load_vars = load_dotenv("../src/.env")
if load_vars:
    print("Variáveis de ambiente carregadas com sucesso")
else:
    print("Arquivo .env não encontrado!")

Variáveis de ambiente carregadas com sucesso


In [11]:
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=tcp:127.0.0.1,1433;"
    "DATABASE=IHM_Testes_2;"
    "UID=sa;"
    f"PWD={os.environ['DB_PASSWORD']};"
    "TrustServerCertificate=yes;"
    "Encrypt=no;"
)

In [12]:
connection_url = f"mssql+pyodbc:///?odbc_connect={quote_plus(connection_string)}"

In [13]:
engine = create_engine(connection_url)

# 3. Pegando todas tabelas

In [14]:
tables = [
    'dados_receitas',
    'fila_batch_ids',
    'fila_paradas',
    'ihms',
    'linhas_producao',
    'logs_registradores',
    'maqteste_status_geral',
    'notificacoes',
    'ordens_servico',
    'paradas',
    'parametros',
    'receitas',
    'registradores',
    'sistemas'
]

In [15]:
dicionario_dfs = dict()
for table in tables:
    df_apoio = pd.read_sql(f"SELECT * FROM {table}", engine)
    if len(df_apoio) > 0:
        dicionario_dfs[table] = df_apoio
    del df_apoio

In [16]:
# for key in dicionario_dfs.keys():
#     print(key)
#     print(dicionario_dfs[key].head())
#     print('------')
    

# 4. Tratando informação

In [17]:
df_interesse = dicionario_dfs['logs_registradores'].merge(dicionario_dfs['ihms'].rename(columns={'id':'id_ihm'})[['id_ihm', 'nome_maquina']], how='left', on='id_ihm')

In [18]:
df_interesse = df_interesse.merge(dicionario_dfs['registradores'].rename(columns={'id':'id_registrador'})[['id_registrador', 'descricao']], how='left', on='id_registrador')

In [19]:
df_interesse = df_interesse[['nome_maquina', 'descricao', 'datahora', 'valor_bruto']]

In [20]:
df_interesse = df_interesse.pivot_table(index=['nome_maquina','datahora'], columns='descricao', values='valor_bruto', aggfunc='first').reset_index()

In [21]:
df_interesse = df_interesse.sort_values('datahora')
df_interesse.reset_index(drop=True, inplace=True)

In [22]:
df_interesse.columns

Index(['nome_maquina', 'datahora', 'engenheiro', 'manutentor', 'motivo_parada',
       'operador', 'produzido', 'reprovado', 'status_maquina',
       'total_produzido'],
      dtype='object', name='descricao')

In [56]:
depara_status_maquina = {
    '0': 'Parada',
    '1': 'Passar Padrão',
    '49': 'Produzindo',
    '4': 'Limpeza'
}

In [57]:
df_interesse['status_maquina'] = df_interesse['status_maquina'].map(depara_status_maquina)

In [23]:
relatorio = "Iniciando relatório..."
colunas_interesse = ['engenheiro', 'manutentor', 'motivo_parada',
       'operador', 'produzido', 'reprovado', 'status_maquina',
       'total_produzido']
dicionario_controle = dict()
for i, row in df_interesse.iterrows():
    relatorio += f"\n\n**{row['datahora']}**" 
    if i > 0:
        relatorio += "\nMudanças de Estado"
    for col in colunas_interesse:
        if col not in dicionario_controle.keys():
            relatorio += f"\n{col} (Estado inicial): {row[col]}"
            dicionario_controle[col] = row[col]
        else:
            if dicionario_controle[col] != row[col]:
                relatorio += f"\n{col}: {row[col]}"
                dicionario_controle[col] = row[col]

In [31]:
# print(relatorio)

In [58]:
df_interesse#.head()

descricao,nome_maquina,datahora,engenheiro,manutentor,motivo_parada,operador,produzido,reprovado,status_maquina,total_produzido
0,MAQ1,2025-11-20 15:06:59.233,0,0,0,7,0,4,Parada,4
1,MAQ1,2025-11-20 15:31:27.607,0,0,1,7,0,4,Passar Padrão,4
2,MAQ1,2025-11-20 15:31:28.673,0,0,0,7,0,4,Parada,4
3,MAQ1,2025-11-20 15:31:53.113,0,0,49,7,0,5,Produzindo,5
4,MAQ1,2025-11-20 15:31:54.723,0,0,49,7,0,5,Produzindo,6
5,MAQ1,2025-11-20 15:31:55.180,0,0,49,7,1,5,Produzindo,6
6,MAQ1,2025-11-20 15:31:57.337,0,0,49,7,1,6,Produzindo,7
7,MAQ1,2025-11-20 15:31:58.187,0,0,49,7,1,7,Produzindo,9
8,MAQ1,2025-11-20 15:31:58.623,0,0,49,7,1,9,Produzindo,10
9,MAQ1,2025-11-20 15:31:59.467,0,0,49,7,1,10,Produzindo,11


In [61]:
df_interesse.shape

(17, 10)

In [91]:
def get_metrics_machine(df_timeline, machine='MAQ1'):
    
    first_register = df_timeline[df_timeline['datahora']==df_timeline['datahora'].min()]
    last_register = df_timeline[df_timeline['datahora']==df_timeline['datahora'].max()]
    status = last_register['status_maquina'].to_list()[0]
    operador = last_register['operador'].to_list()[0]
    manutentor = last_register['manutentor'].to_list()[0]
    produzido = last_register['produzido'].to_list()[0]
    reprovado = last_register['reprovado'].to_list()[0]
    total = last_register['total_produzido'].to_list()[0]
    
    # OEE = Disponibilidade * Performance * Qualidade
    
    # Disponibilidade = Tempo produzido / Tempo programado para produzir
    lista_produzido = []
    status_antigo = ""
    inicio = None
    fim = None
    for i, row in df_timeline.iterrows():
        if status_antigo != 'Produzindo' and row['status_maquina']=='Produzindo':
            inicio = row['datahora']
        elif (status_antigo == 'Produzindo' and row['status_maquina']!='Produzindo') or (status_antigo == 'Produzindo' and row['status_maquina']=='Produzindo' and row['datahora']==last_register['datahora'].to_list()[0]):
            fim = row['datahora']
        if inicio and fim:
            lista_produzido.append((inicio, fim))
            inicio = None
            fim = None            
        status_antigo = row['status_maquina']
    tempo_produzido = sum([y.total_seconds() for y in [x[1] - x[0] for x in lista_produzido]])
    tempo_programado = (last_register['datahora'].to_list()[0] - first_register['datahora'].to_list()[0]).total_seconds()
    disponibilidade = tempo_produzido / tempo_programado
        
    # Performance = Produção Real / Produção Teórica
    meta = (tempo_programado // 1) # Considerando que a cada 1 s é feito uma peça
    performance = int(total) / meta
    
    # Qualidade = Peça Boas / Total de peças
    qualidade = int(produzido) / int(total)
    
    oee = disponibilidade * performance * qualidade
    # OEE, Qualidade, Eficiencia, Meta, Acumulado, Operador, Manutentor, Status
    
    
    return {
        'status': status,
        'oee': round(100 * oee, 2),
        'eficiencia': performance, # eficiencia é isso mesmo?
        'qualidade': round(100 * qualidade, 2),
        'meta': meta,
        'produzido': produzido,
        'reprovado': reprovado,
        'total': total,
        'operador': operador,
        'manutentor': manutentor
    }
    

In [92]:
get_metrics_machine(df_interesse)

{'status': 'Parada',
 'oee': 0.0,
 'eficiencia': 0.004561003420752566,
 'qualidade': 16.67,
 'meta': 2631.0,
 'produzido': '2',
 'reprovado': '10',
 'total': '12',
 'operador': '7',
 'manutentor': '0'}